In [2]:
MODE = "train"                # "test" (teammate C) or "train" (teammate D)
SMOKE = False                 # True: quick code check. False: full run (Save & Run All)
SMOKE_QUERIES = 2_000        # smoke: queries per country
SMOKE_POOL = 200_000         # smoke: pool records per country (recall NOT meaningful)
 
DATA_ROOT = "/kaggle/input/datasets/mahimasunilbachhav/amazon-ml-challenge/student_resource/dataset"
INPUT_ROOT = "/kaggle/input"
WORK = "/kaggle/working"
 
K8, K2T = 100, 30
DF_CAP = 2_000
MAX_ADDR_TOKENS = 8
FIELDS = {"name": ["n", "nb", "nf", "k", "ks"], "addr": ["a", "d", "ap"]}
TYPE_W = {"n": 1.0, "nb": 1.0, "nf": 1.0, "k": 1.0, "ks": 0.7, "a": 1.0, "d": 1.0, "ap": 1.0}
FIELD_W = {"name": 1.0, "addr": 1.5}      # "address-heavy": best in the probe
Q_CHUNK = 2_000              # queries per search task
TOK_CHUNK = 100_000          # records per token-building task
DIM_L2T, MIN_SIM = 256, 0.10
TFIDF_FIT_ROWS, SVD_FIT_ROWS, TFIDF_CHUNK = 1_000_000, 300_000, 250_000
Q_BATCH = 1024
SEED = 42

In [3]:
import gc
import hashlib
import json
import shutil
import subprocess
import sys
import time
import zipfile
from collections import Counter, defaultdict
from pathlib import Path
 
import numpy as np
import pandas as pd
import psutil
import lightgbm as lgb
from tqdm.auto import tqdm
 
try:
    from rapidfuzz.process import cpdist
except ImportError:
    print("installing/upgrading rapidfuzz (needs Internet ON) ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "rapidfuzz"], check=True)
    try:
        from rapidfuzz.process import cpdist
    except ImportError:
        raise SystemExit("rapidfuzz was upgraded: restart the session, then rerun this cell.")
from rapidfuzz import fuzz
from rapidfuzz.distance import JaroWinkler
 
DATA_ROOT, WORK, INPUT_ROOT = Path(DATA_ROOT), Path(WORK), Path(INPUT_ROOT)
T_START = time.time()
TIMES = {}                     # stage -> [seconds, scalable?]
 
 
def log(msg):
    print(f"  [{(time.time()-T_START)/60:5.1f} min] {msg}", flush=True)
 
 
def mem(tag=""):
    rss = psutil.Process().memory_info().rss / 1e9
    print(f"    [MEM] {tag:<28} RAM={rss:5.1f} GB  free={psutil.virtual_memory().available/1e9:5.1f} GB",
          flush=True)
 
 
def bar(it=None, **kw):
    kw.setdefault("mininterval", 10)
    return tqdm(it, **kw)
 
 
def timed(stage, t0, scalable=True):
    """Accumulate wall time per stage. scalable: True = grows with the number of
    pairs, False = fixed cost, None = done once and reused by the full run."""
    TIMES.setdefault(stage, [0.0, scalable])[0] += time.time() - t0
 
 
def hash_bucket(eid):
    """Deterministic 0-9999 bucket; identical to the EDA / Notebook 2 split."""
    return int(hashlib.md5(eid.encode()).hexdigest(), 16) % 10000
 
 
def locate(marker):
    """All files named `marker` under INPUT_ROOT, plus those inside any .zip under
    INPUT_ROOT. Each zip holding the marker is unpacked once into
    WORK/_unzipped/<zip name>/ (Kaggle may keep uploaded zips packed)."""
    root_ok = INPUT_ROOT.exists()
    hits = sorted(INPUT_ROOT.rglob(marker)) if root_ok else []
    for z in (sorted(INPUT_ROOT.rglob("*.zip")) if root_ok else []):
        dest = WORK / "_unzipped" / z.stem
        done = dest / ".unpacked"
        if not done.exists():
            with zipfile.ZipFile(z) as zf:
                if not any(n.endswith("/" + marker) or n == marker for n in zf.namelist()):
                    continue
                log(f"unpacking {z.name} -> {dest}")
                zf.extractall(dest)
            done.touch()
        hits += sorted(dest.rglob(marker))
    return hits
 
 
def project(full_factor, label):
    """Print measured stage times and their projection to the full run."""
    print(f"\n  RUNTIME PROJECTION ({label})")
    print(f"    {'stage':<30}{'smoke (s)':>11}{'full (min)':>12}")
    total = 0.0
    for stage, (sec, scalable) in TIMES.items():
        # scalable None = already done and reused by the full run
        full = 0.0 if scalable is None else sec * (full_factor if scalable else 1.0)
        total += full
        print(f"    {stage:<30}{sec:>11.1f}{full/60:>12.1f}")
    print(f"    {'TOTAL':<30}{'':>11}{total/60:>12.1f}")
    return total
 

In [4]:
# Text cleaning + transliteration (shared with the pipeline)
# ---------------------------------------------------------------------------
# Same cleaning as every other stage.
# ---------------------------------------------------------------------------
COLS = ["entity_id", "business_name", "business_address", "country"]
 
 
def load_src(path):
    """Read a source TSV. sep='\t' is mandatory: fields contain commas."""
    return pd.read_csv(path, sep="\t", dtype=str, names=COLS, header=0, keep_default_na=False)
 
 
# ---- cleaning: copied verbatim from Notebook 2 so both stages agree ----------
LEGAL_RE = (r"\b(llc|l\.l\.c|inc|corp|corporation|ltd|limited|pvt|private|"
            r"llp|lp|pllc|sas|sarl|eurl|sasu|snc|company)\b")
 
ADDR_ABBR = [
    (r"\brd\b", "road"), (r"\bst\b", "street"), (r"\bdr\b", "drive"),
    (r"\bave\b", "avenue"), (r"\bblvd\b", "boulevard"), (r"\bln\b", "lane"),
    (r"\bct\b", "court"), (r"\bcir\b", "circle"), (r"\bhno\b", "house number"),
    (r"\bno\b", "number"), (r"\bopp\b", "opposite"), (r"\bnr\b", "near"),
    (r"\bdist\b", "district"), (r"\bflr\b", "floor"),
]
 
 
def clean_text(s):
    """Shared cleaning. Non-Latin scripts are dropped here; L3 reads raw text."""
    s = s.fillna("").astype(str).str.lower()
    s = (s.str.normalize("NFKD")
          .str.encode("ascii", errors="ignore")
          .str.decode("ascii"))
    s = s.str.replace(r"##\s*", "", regex=True)
    s = s.str.replace(r"\bwww\.", "", regex=True)
    s = s.str.replace(r"\.(com|net|org|co|in|fr|uk|biz|info|io)\b", "", regex=True)
    s = s.str.replace(r"^[\s\-\.\,\!\@\#\$\%\^\&\*\>\<\'\"/\\]+", "", regex=True)
    s = s.str.replace("&", " and ", regex=False)
    s = s.str.replace(r"\bn/?a\b", " ", regex=True)
    s = s.str.replace(r"[^\w\s]", " ", regex=True)
    return s.str.replace(r"\s+", " ", regex=True).str.strip()
 
 
def clean_name(s):
    """Name: shared cleaning + legal suffixes + repeated words (FH FH -> FH)."""
    s = clean_text(s)
    s = s.str.replace(LEGAL_RE, " ", regex=True)
    s = s.str.replace(r"\b(\w+)\s+\1\b", r"\1", regex=True)
    return s.str.replace(r"\s+", " ", regex=True).str.strip()
 
 
def clean_addr(s):
    """Address: shared cleaning + abbreviation expansion."""
    s = clean_text(s)
    for pat, rep in ADDR_ABBR:
        s = s.str.replace(pat, rep, regex=True)
    return s.str.replace(r"\s+", " ", regex=True).str.strip()
 
 
# ---- extra text columns for Stage 2 -------------------------------------------
# Street-type words and French articles are skipped when forming the address key,
# so "15 Rue de la Hoxie" -> "15_hoxie" (not "15_rue"), matching "351 Hoxie Avenue"
# style keys. This matters most for France, which training never sees.
SKIP_AFTER_NUMBER = {
    "road", "street", "drive", "avenue", "boulevard", "lane", "court", "circle",
    "way", "highway", "rue", "impasse", "allee", "chemin", "place", "route", "quai",
    "cours", "square", "de", "du", "des", "la", "le", "les", "l", "d",
}
 
 
def addr_key_string(caddr):
    """House number + the first following token that isn't a street word/article.
    Returned as a space-joined, sorted string so rapidfuzz can compare two of them."""
    toks = caddr.split()
    keys = set()
    for i, a in enumerate(toks):
        if a.isdigit():
            a2 = a.lstrip("0")
            j = i + 1
            while j < len(toks) and toks[j] in SKIP_AFTER_NUMBER:
                j += 1
            if a2 and j < len(toks):
                b = toks[j].lstrip("0") if toks[j].isdigit() else toks[j]
                if b:
                    keys.add(a2 + "_" + b)
    return " ".join(sorted(keys))
 
 
def prepare_texts(df):
    """All per-record text columns Stage 2 needs, computed once."""
    out = pd.DataFrame({"entity_id": df["entity_id"].values})
    out["cname"] = clean_name(df["business_name"]).values
    out["caddr"] = clean_addr(df["business_address"]).values
    out["digits"] = out["caddr"].str.findall(r"\b\d+\b").str.join(" ")
    out["nospace"] = out["cname"].str.replace(" ", "", regex=False)
    out["akeys"] = [addr_key_string(a) for a in out["caddr"]]
    out["nonlatin"] = np.array([any(ord(ch) > 0x024F for ch in s)
                                for s in df["business_name"]], dtype=np.int8)
    return out
 
 
 
 
# ---- v2: transliteration, legal form, phonetic skeleton (shared) -------------
import re as _re
import unicodedata as _ud
try:
    from indic_transliteration import sanscript as _sanscript
    from indic_transliteration.sanscript import transliterate as _translit
except ImportError:
    print("installing indic_transliteration (MIT licence; needs Internet ON) ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "indic_transliteration"], check=True)
    from indic_transliteration import sanscript as _sanscript
    from indic_transliteration.sanscript import transliterate as _translit
 
_BLOCK_SCRIPT = {0x0900: _sanscript.DEVANAGARI, 0x0980: _sanscript.BENGALI,
                 0x0A00: _sanscript.GURMUKHI, 0x0A80: _sanscript.GUJARATI,
                 0x0B00: _sanscript.ORIYA, 0x0B80: _sanscript.TAMIL,
                 0x0C00: _sanscript.TELUGU, 0x0C80: _sanscript.KANNADA,
                 0x0D00: _sanscript.MALAYALAM}
_LEGAL_TOKENS = {"pvt", "private", "pr", "pra", "ltd", "limited", "li", "llp", "pllc", "llc",
                 "lp", "inc", "incorporated", "corp", "corporation", "co", "company", "sas",
                 "sasu", "sarl", "eurl", "snc", "sa", "public"}
_LEGAL_PREFIX = ("prai", "pirai", "limi", "elel")    # transliterated private / limited / LLP
_FORM_KEYS = (("LLP", {"llp"}), ("PLLC", {"pllc"}), ("LLC", {"llc"}), ("LP", {"lp"}),
              ("INC", {"inc", "incorporated"}), ("CORP", {"corp", "corporation"}),
              ("SASU", {"sasu"}), ("SAS", {"sas"}), ("SARL", {"sarl"}), ("EURL", {"eurl"}),
              ("SNC", {"snc"}), ("SA", {"sa"}))
_SKEL_FOLD = (("ph", "f"), ("th", "t"), ("sh", "s"), ("kh", "k"), ("gh", "g"), ("bh", "b"),
              ("dh", "d"), ("ch", "c"), ("ck", "k"), ("c", "k"), ("q", "k"), ("x", "ks"),
              ("z", "s"), ("w", "v"), ("j", "g"), ("y", ""), ("v", "b"))   # Bengali renders B as v
 
 
def latin_tokens(name):
    """Lowercase Latin tokens of a name. Indic-script words are transliterated
    (ITRANS), accents removed, dots dropped (L.L.C -> llc), other punctuation
    splits words (Housing-Network -> housing network)."""
    s = str(name)
    s = _re.sub(r"\.(com|net|org|co|in|fr|uk|biz|info|io)\b", " ", s, flags=_re.I)
    s = s.replace(".", "")
    fast = all(ord(ch) <= 0x024F for ch in s)
    if fast:
        s = _re.sub(r"[^\w\s]", " ", s)
    else:
        # Indic vowel signs / virama are not \w, so only replace real punctuation
        # and symbols (Unicode categories P* and S*).
        s = "".join(" " if _ud.category(ch)[0] in "PS" else ch for ch in s)
    out = []
    for w in s.split():
        indic = False
        if not fast:
            blk = next((b for b in _BLOCK_SCRIPT if any(b <= ord(c) < b + 0x80 for c in w)), None)
            if blk is not None:
                indic = True
                try:
                    w = _translit(w, _BLOCK_SCRIPT[blk], _sanscript.ITRANS)
                except Exception:
                    pass
                if blk == 0x0B80:                     # Tamil: library leaves some letters
                    w = w.replace("\u0BA9", "n").replace("\u0BB1", "r")
        w = _ud.normalize("NFKD", w).encode("ascii", "ignore").decode().lower()
        if indic and blk == 0x0B80:
            # Tamil has no aspirated/voiced contrast; the library writes ச க ப ட/த
            # as jh gh bh dh — fold them to s k p t.
            for a, b in (("jh", "s"), ("gh", "k"), ("bh", "p"), ("dh", "t")):
                w = w.replace(a, b)
        w = _re.sub(r"[^a-z0-9]", "", w)
        if indic:
            w = _re.sub(r"([bcdfghjklmnpqrstvwxyz])a$", r"\1", w)    # inherent final 'a'
        if w:
            out.append(w)
    return out
 
 
def legal_form(toks):
    """Canonical legal form: PVT (private limited), PLC, LTD, LLP, LLC, INC, ...
    Spelling variants map to one code (Pvt Ltd = Private Limited = प्रा. लि.)."""
    s = set(toks)
    has_pvt = bool(s & {"pvt", "private", "pr", "pra"}) or any(t.startswith(("prai", "pirai")) for t in toks)
    has_ltd = bool(s & {"ltd", "limited", "li"}) or any(t.startswith("limi") for t in toks)
    if has_pvt:
        return "PVT"
    if "public" in s and has_ltd:
        return "PLC"
    if any(t.startswith("elel") for t in toks):
        return "LLP"
    for code, keys in _FORM_KEYS:
        if s & keys:
            return code
    if has_ltd:
        return "LTD"
    if s & {"co", "company"}:
        return "CO"
    return ""

In [5]:
# L8 token builder (verified in the probe)
# ---------------------------------------------------------------------------
# Typed tokens with exact ids, built in parallel.
# ---------------------------------------------------------------------------
# ---- L8 token builder ---------------------------------------------------------
# Tokens get EXACT column ids: each token string -> 64-bit fingerprint (Python
# hash; identical in every forked worker of this session) -> numbered exactly
# with pandas. No shared hash buckets, so rare tokens stay rare.
import os
import multiprocessing as _mp
import scipy.sparse as sp
from concurrent.futures import ProcessPoolExecutor
 
TYPES = ["n", "nb", "nf", "k", "ks", "a", "d", "ap"]
_FOLD = tuple(_SKEL_FOLD)
 
 
def skel_word(w):
    """Phonetic skeleton of one word (m->n BEFORE collapsing repeats: fixes the
    Phase B order bug, so 'investments' -> 'nvstnts')."""
    for a, b in _FOLD:
        w = w.replace(a, b)
    w = _re.sub(r"[aeiou]", "", w).replace("m", "n")
    return _re.sub(r"(.)\1+", r"\1", w)
 
 
def _is_legal(w):
    return w in _LEGAL_TOKENS or w.startswith(_LEGAL_PREFIX)
 
 
def record_tokens(name_words, raw_name, translation, caddr):
    """Typed tokens of one record (sets, so repeats count once)."""
    T = {t: set() for t in TYPES}
    words = [w for w in name_words if len(w) >= 2]
    T["n"].update(words)
    T["nb"].update(f"{a}_{b}" for a, b in zip(words, words[1:]))
    full = "".join(words)
    if len(full) >= 4:
        T["nf"].add(full)
    src = [w for w in latin_tokens(raw_name) if not _is_legal(w)] + words
    if translation:
        src += [w for w in latin_tokens(translation) if not _is_legal(w)]
    for w in src:
        s = skel_word(w)
        if len(s) >= 2:
            T["k"].add(s)
        if len(s) >= 3:
            T["ks"].add("".join(sorted(s)))
    atoks = caddr.split()
    for w in atoks:
        if w.isdigit():
            if w.lstrip("0"):
                T["d"].add(w.lstrip("0"))
        elif len(w) >= 2:
            T["a"].add(w)
    firsts = list(dict.fromkeys(w.lstrip("0") or w if w.isdigit() else w for w in atoks))[:MAX_ADDR_TOKENS]
    for i in range(len(firsts)):
        for j in range(i + 1, len(firsts)):
            a, b = sorted((firsts[i], firsts[j]))
            T["ap"].add(f"{a}|{b}")
    return T
 
 
def _token_task(args):
    """Worker: records -> per token type (tokens per record, flat fingerprints)."""
    names, addrs, trans = args
    base = pd.Series([t if t else n for n, t in zip(names, trans)])
    nw = clean_name(base).str.split().tolist()
    ca = clean_addr(pd.Series(addrs)).tolist()
    per = {t: [] for t in TYPES}
    for w, n, tr, a in zip(nw, names, trans, ca):
        T = record_tokens(w, n, tr, a)
        for t in TYPES:
            per[t].append(T[t])
    out = {}
    for t in TYPES:
        lens = np.fromiter((len(x) for x in per[t]), dtype=np.int32, count=len(per[t]))
        flat = np.fromiter((hash(x) for s in per[t] for x in s), dtype=np.int64, count=int(lens.sum()))
        out[t] = (lens, flat)
    return out
 
 
def build_tokens(names, addrs, trans, desc):
    """Typed token fingerprints for many records, in parallel over CPU cores."""
    tasks = [(names[i:i + TOK_CHUNK], addrs[i:i + TOK_CHUNK], trans[i:i + TOK_CHUNK])
             for i in range(0, len(names), TOK_CHUNK)]
    parts = []
    ctx = _mp.get_context("fork")            # same hash seed in every worker
    with ProcessPoolExecutor(max_workers=os.cpu_count() or 2, mp_context=ctx) as ex:
        for r in bar(ex.map(_token_task, tasks), total=len(tasks), desc=desc, unit="chunk"):
            parts.append(r)
    return {t: (np.concatenate([p[t][0] for p in parts]), np.concatenate([p[t][1] for p in parts]))
            for t in TYPES}
 
 
def to_csr(lens, flat, vocab):
    """Binary record x token matrix; tokens missing from `vocab` are dropped."""
    codes = vocab.get_indexer(flat)
    rows = np.repeat(np.arange(len(lens), dtype=np.int64), lens)
    keep = codes >= 0
    X = sp.csr_matrix((np.ones(int(keep.sum()), np.float32), (rows[keep], codes[keep])),
                      shape=(len(lens), len(vocab)))
    X.sum_duplicates()
    X.data[:] = 1.0
    return X
 

In [6]:
# %% [CELL 5] L8 parallel search + GPU helpers for L2t
# ---------------------------------------------------------------------------
# Search runs on all CPU cores; L2t uses the GPU.
# ---------------------------------------------------------------------------
# ---- L8 parallel search ----------------------------------------------------------
# Workers are forked, so they read the index (_IX) without copying it.
_IX = {}
 
 
def _search_task(bounds):
    """Top-K L8 candidates for queries r0..r1 -> (q, p, score, name_sim, addr_sim)."""
    r0, r1 = bounds
    PT, Qs, fpn, fqn = _IX["PT"], _IX["Qs"], _IX["fpn"], _IX["fqn"]
    FM = {}
    for f, ts in FIELDS.items():
        M = None
        for t in ts:
            part = (Qs[t][r0:r1] @ PT[t]) * (TYPE_W[t] ** 2)
            M = part if M is None else M + part
        M = M.tocsr()
        M.sort_indices()
        rows = np.repeat(np.arange(r1 - r0), np.diff(M.indptr))
        M.data /= np.maximum(fqn[f][r0:r1][rows] * fpn[f][M.indices], 1e-12)
        FM[f] = M
    M = (FM["name"] * FIELD_W["name"] + FM["addr"] * FIELD_W["addr"]).tocsr()
    M.sort_indices()
    rows = np.repeat(np.arange(r1 - r0), np.diff(M.indptr))
    den = sum(FIELD_W[f] * (fqn[f][r0:r1][rows] > 0) * (fpn[f][M.indices] > 0) for f in FIELDS)
    M.data /= np.maximum(den, 1e-12)
    oq, op, osc, on, oa = [], [], [], [], []
    for i in range(r1 - r0):
        s, e = M.indptr[i], M.indptr[i + 1]
        if s == e:
            continue
        cols, vals = M.indices[s:e], M.data[s:e]
        if e - s > K8:
            sel = np.argpartition(-vals, K8)[:K8]
            cols, vals = cols[sel], vals[sel]
        fv = {}
        for f in FIELDS:                       # this pair's own name / address similarity
            fs, fe = FM[f].indptr[i], FM[f].indptr[i + 1]
            fi, fd = FM[f].indices[fs:fe], FM[f].data[fs:fe]
            pos = np.searchsorted(fi, cols)
            pos = np.minimum(pos, max(len(fi) - 1, 0))
            hit = (len(fi) > 0) & (fi[pos] == cols) if len(fi) else np.zeros(len(cols), bool)
            fv[f] = np.where(hit, fd[pos] if len(fd) else 0.0, 0.0)
        oq.append(np.full(len(cols), r0 + i, dtype=np.int64))
        op.append(cols.astype(np.int64)); osc.append(vals)
        on.append(fv["name"]); oa.append(fv["addr"])
    if not oq:
        z = np.zeros(0)
        return z.astype(np.int64), z.astype(np.int64), z, z, z
    return tuple(np.concatenate(x) for x in (oq, op, osc, on, oa))
 
 
def l8_search(nq):
    tasks = [(r0, min(nq, r0 + Q_CHUNK)) for r0 in range(0, nq, Q_CHUNK)]
    parts = []
    ctx = _mp.get_context("fork")
    with ProcessPoolExecutor(max_workers=os.cpu_count() or 2, mp_context=ctx) as ex:
        for r in bar(ex.map(_search_task, tasks), total=len(tasks), desc="L8 search", unit="chunk"):
            parts.append(r)
    return tuple(np.concatenate([p[k] for p in parts]) for k in range(5))
 
 
import torch
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize
 
 
def purge():
    gc.collect()
    torch.cuda.empty_cache()
 
 
# ---- GPU exact top-k (same algorithm as Notebook 2) ---------------------------
def gpu_topk(Q, P, k, desc, q_batch=Q_BATCH, p_chunk=1_000_000):
    """Exact inner-product top-k on the GPU (fp16); -1 where score < MIN_SIM."""
    k = min(k, P.shape[0])
    P_t = torch.from_numpy(np.ascontiguousarray(P, dtype=np.float16)).to(DEV)
    I = np.empty((Q.shape[0], k), dtype=np.int32)
    S = np.empty((Q.shape[0], k), dtype=np.float16)
    for i in bar(range(0, Q.shape[0], q_batch), desc=desc, unit="batch"):
        q = torch.from_numpy(np.ascontiguousarray(Q[i:i + q_batch], dtype=np.float16)).to(DEV)
        best_v = best_i = None
        for j in range(0, P_t.shape[0], p_chunk):
            s = q @ P_t[j:j + p_chunk].T
            v, ix = torch.topk(s, min(k, s.shape[1]), dim=1)
            ix += j
            if best_v is None:
                best_v, best_i = v, ix
            else:
                cv, ci = torch.cat([best_v, v], dim=1), torch.cat([best_i, ix], dim=1)
                best_v, sel = torch.topk(cv, k, dim=1)
                best_i = torch.gather(ci, 1, sel)
            del s
        n = q.shape[0]
        I[i:i + n] = best_i.cpu().numpy()
        S[i:i + n] = best_v.cpu().numpy()
    del P_t
    purge()
    I[S < MIN_SIM] = -1
    return I, S
 
 
def fit_tfidf_svd(p_txt, dim, ngram):
    """Char n-gram TF-IDF (vocabulary from a sample) + SVD fitted on a sample."""
    rng = np.random.default_rng(SEED)
    idx = rng.choice(len(p_txt), min(TFIDF_FIT_ROWS, len(p_txt)), replace=False)
    vec = TfidfVectorizer(analyzer="char_wb", ngram_range=ngram, min_df=3, max_df=0.5,
                          max_features=40_000, sublinear_tf=True, dtype=np.float32)
    X = vec.fit_transform([p_txt[i] for i in idx])
    dim = min(dim, X.shape[1] - 1)
    sidx = rng.choice(X.shape[0], min(SVD_FIT_ROWS, X.shape[0]), replace=False)
    svd = TruncatedSVD(n_components=dim, algorithm="randomized", n_iter=4,
                       random_state=SEED).fit(X[sidx])
    return vec, svd
 
 
def tfidf_transform(vec, svd, texts, desc):
    out = np.empty((len(texts), svd.n_components), dtype=np.float16)
    for i in bar(range(0, len(texts), TFIDF_CHUNK), desc=desc, unit="chunk"):
        out[i:i + TFIDF_CHUNK] = normalize(svd.transform(vec.transform(texts[i:i + TFIDF_CHUNK])))
    return out

In [ ]:
# import zipfile, glob
# for z in glob.glob("/kaggle/input/**/results.zip", recursive=True):
#     names = [n for n in zipfile.ZipFile(z).namelist() if n.endswith((".npz", ".json"))]
#     print(z, "\n ", names[:6])

In [7]:
import torch
HAS_GPU = torch.cuda.is_available()
assert MODE in ("test", "train")
print("GPU:", torch.cuda.get_device_name(0) if HAS_GPU else "NONE — L2t needs a GPU (set Accelerator: GPU T4)")
CDIR = {f.parent.name: f.parent for f in locate("qids.npy") if f.parent.parent.name == f"ckpt_{MODE}"}
assert CDIR, f"ckpt_{MODE} not found: attach the {MODE} blocking zips"
tr_files = sorted(INPUT_ROOT.rglob(f"translations_{MODE}.parquet")) + sorted(WORK.glob(f"translations_{MODE}.parquet"))
assert tr_files, f"translations_{MODE}.parquet not found: attach the translation pre-step ({MODE}) output"
t = pd.read_parquet(tr_files[0], columns=["entity_id", "translation"])
TRANS = dict(zip(t["entity_id"], t["translation"]))
del t
COUNTRIES = sorted(CDIR)
OUT = WORK / f"ckpt_c_{MODE}{'_smoke' if SMOKE else ''}"
OUT.mkdir(parents=True, exist_ok=True)
recs = pd.concat([load_src(DATA_ROOT / MODE / f"{MODE}_source{i}.tsv") for i in (1, 2, 3)],
                 ignore_index=True).set_index("entity_id")
GT = {}
if MODE == "train":
    gt = pd.read_csv(DATA_ROOT / "train" / "train_ground_truth.tsv", sep="\t", dtype=str,
                     names=["s1", "m"], header=0, keep_default_na=False)
    GT = {s: [x for x in m.split(",") if x] for s, m in zip(gt["s1"], gt["m"])}
    del gt
BDIR = {f.parent.name: f.parent for f in locate("pairs_b.npz") if f.parent.parent.name == f"ckpt_b_{MODE}"}
print("=" * 70)
print(f"PHASE C  MODE={MODE}  {'SMOKE RUN' if SMOKE else 'FULL RUN'}  countries={COUNTRIES}")
print(f"  translations: {len(TRANS):,}   Phase B output for recall report: {'yes' if BDIR else 'no'}")
print("=" * 70)
 
 
def merge_scored(nq, npool, parts):
    """Union of (q, p, {col: score}) parts -> sorted unique pairs, one column per score."""
    keys = np.concatenate([q.astype(np.int64) * npool + p for q, p, _ in parts])
    uk, inv = np.unique(keys, return_inverse=True)
    out = {"q": (uk // npool).astype(np.int32), "p": (uk % npool).astype(np.int32)}
    off = 0
    cols = sorted({c for _, _, s in parts for c in s})
    for c in cols:
        out[c] = np.zeros(len(uk), dtype=np.float32)
    for q, p, s in parts:
        idx = inv[off:off + len(q)]
        for c, v in s.items():
            np.maximum.at(out[c], idx, v.astype(np.float32))
        off += len(q)
    for c in cols:
        out[c] = out[c].astype(np.float16)
    return out
 
 


GPU: Tesla T4
PHASE C  MODE=train  FULL RUN  countries=['India', 'US']
  translations: 752,869   Phase B output for recall report: yes


In [8]:
# %% [CELL 7] Per country: L8 + L2t (checkpointed per country)
full_q = used_q = full_p = used_p = 0
for c in COUNTRIES:
    qids = np.load(CDIR[c] / "qids.npy", allow_pickle=True)
    pids = np.load(CDIR[c] / "pids.npy", allow_pickle=True)
    nq = min(SMOKE_QUERIES, len(qids)) if SMOKE else len(qids)
    npool = min(SMOKE_POOL, len(pids)) if SMOKE else len(pids)
    full_q += len(qids); used_q += nq; full_p += len(pids); used_p += npool
    qids, pids = qids[:nq], pids[:npool]
    if (OUT / c / "pairs_c.npz").exists():
        log(f"[{c}] already done — skipping")
        continue
    (OUT / c).mkdir(parents=True, exist_ok=True)
    log(f"[{c}] {nq:,} queries vs {npool:,} pool records")
 
    # ---- L8 index ---------------------------------------------------------------
    t0 = time.time()
    P_rec, Q_rec = recs.reindex(pids), recs.reindex(qids)
    Praw = build_tokens(P_rec["business_name"].tolist(), P_rec["business_address"].tolist(),
                        [TRANS.get(x, "") for x in pids], f"L8 pool {c}")
    Qraw = build_tokens(Q_rec["business_name"].tolist(), Q_rec["business_address"].tolist(),
                        [""] * nq, f"L8 queries {c}")
    PT, Qs, sq_p, sq_q = {}, {}, {}, {}
    for t in TYPES:
        vocab = pd.Index(pd.unique(Praw[t][1]))
        P_t, Q_t = to_csr(*Praw[t], vocab), to_csr(*Qraw[t], vocab)
        del vocab
        df = np.bincount(P_t.indices, minlength=P_t.shape[1])
        idf = np.where((df > 0) & (df <= DF_CAP), np.log(npool / np.maximum(df, 1)), 0.0).astype(np.float32)
        P_t.data = idf[P_t.indices]; P_t.eliminate_zeros()
        Q_t.data = idf[Q_t.indices]; Q_t.eliminate_zeros()
        sq_p[t] = np.asarray(P_t.multiply(P_t).sum(1)).ravel()
        sq_q[t] = np.asarray(Q_t.multiply(Q_t).sum(1)).ravel()
        PT[t], Qs[t] = P_t.T.tocsr(), Q_t
        del P_t
    del Praw, Qraw
    gc.collect()
    _IX.clear()
    _IX.update(PT=PT, Qs=Qs,
               fpn={f: np.sqrt(sum(TYPE_W[t] ** 2 * sq_p[t] for t in ts)) for f, ts in FIELDS.items()},
               fqn={f: np.sqrt(sum(TYPE_W[t] ** 2 * sq_q[t] for t in ts)) for f, ts in FIELDS.items()})
    timed("L8 index", t0, scalable="pool")
    mem(f"{c} L8 index")
 
    # ---- L8 search (parallel) -------------------------------------------------------
    t0 = time.time()
    q8, p8, s8, n8, a8 = l8_search(nq)
    _IX.clear(); del PT, Qs
    gc.collect()
    timed("L8 search", t0, scalable="both")
    parts = [(q8, p8, {"l8": s8, "l8n": n8, "l8a": a8})]
 
    # ---- L2t: translated non-Latin records only (GPU) ---------------------------
    t0 = time.time()
    sub = np.flatnonzero(np.fromiter((x in TRANS for x in pids), dtype=bool, count=npool))
    if HAS_GPU and len(sub) > 50:
        ptxt = (clean_name(pd.Series([TRANS[x] for x in pids[sub]])) + " " +
                clean_addr(P_rec["business_address"].iloc[sub].reset_index(drop=True))).tolist()
        qtxt = (clean_name(Q_rec["business_name"].reset_index(drop=True)) + " " +
                clean_addr(Q_rec["business_address"].reset_index(drop=True))).tolist()
        vec, svd = fit_tfidf_svd(ptxt, DIM_L2T, (3, 4))
        Pv = tfidf_transform(vec, svd, ptxt, f"L2t pool {c}")
        Qv = tfidf_transform(vec, svd, qtxt, f"L2t queries {c}")
        I, S = gpu_topk(Qv, Pv, K2T, f"L2t search {c}")
        ok = I >= 0
        parts.append((np.nonzero(ok)[0].astype(np.int64), sub[I[ok]].astype(np.int64),
                      {"l2t": S[ok].astype(np.float32)}))
        del Pv, Qv, vec, svd
        purge()
        log(f"[{c}] L2t over {len(sub):,} translated records")
    timed("L2t", t0, scalable="pool")
 
    # ---- merge, label, save -----------------------------------------------------
    t0 = time.time()
    out = merge_scored(nq, npool, parts)
    for col in ("l8", "l8n", "l8a", "l2t"):
        out.setdefault(col, np.zeros(len(out["q"]), dtype=np.float16))
    stats = {"queries": nq, "pool": npool, "pairs": int(len(out["q"])),
             "pairs_per_query": float(len(out["q"]) / max(nq, 1)),
             "l8_pairs": int((out["l8"] > 0).sum()), "l2t_pairs": int((out["l2t"] > 0).sum())}
    if MODE == "train":
        ppos = pd.Series(np.arange(npool), index=pids)
        tq, tp, n_true = [], [], np.zeros(nq, dtype=np.int64)
        for i in range(nq):
            ms = GT.get(qids[i], ())
            n_true[i] = len(ms)
            for m in ms:
                j = ppos.get(m)
                if j is not None:
                    tq.append(i); tp.append(j)
        true_keys = np.unique(np.array(tq, dtype=np.int64) * npool + np.array(tp, dtype=np.int64))
        ckeys = out["q"].astype(np.int64) * npool + out["p"]
        out["label"] = np.isin(ckeys, true_keys).astype(np.int8)
        d_old = np.load(CDIR[c] / "pairs.npz")
        cut = int(np.searchsorted(d_old["q"], nq))
        ok = d_old["p"][:cut] < npool
        old_keys = d_old["q"][:cut][ok].astype(np.int64) * npool + d_old["p"][:cut][ok]
        del d_old
        b_keys = np.zeros(0, np.int64)
        if c in BDIR:
            d_b = np.load(BDIR[c] / "pairs_b.npz")
            cut = int(np.searchsorted(d_b["q"], nq))
            ok = d_b["p"][:cut] < npool
            b_keys = d_b["q"][:cut][ok].astype(np.int64) * npool + d_b["p"][:cut][ok]
            del d_b
        qb = np.array([hash_bucket(x) for x in qids], dtype=np.int16)
 
        def recall(keys, sel):
            tk = true_keys[sel[true_keys // npool]]
            return float(np.isin(tk, keys).sum() / max(n_true[sel].sum(), 1))
 
        ob = np.union1d(old_keys, b_keys)
        allk = np.union1d(ob, ckeys)
        A, H = np.ones(nq, bool), qb < 500
        stats.update({"recall_old": recall(old_keys, A), "recall_old_b": recall(ob, A),
                      "recall_all": recall(allk, A), "recall_old_b_heldout": recall(ob, H),
                      "recall_all_heldout": recall(allk, H),
                      "recall_l8_alone": recall(ckeys[out["l8"] > 0], A),
                      "recall_l2t_alone": recall(ckeys[out["l2t"] > 0], A),
                      "new_pairs_per_query": float((~np.isin(ckeys, ob)).sum() / max(nq, 1))})
        del old_keys, b_keys, ob, allk
    np.savez(OUT / c / "pairs_c.npz", **out)
    json.dump(stats, open(OUT / c / "stats_c.json", "w"), indent=2)
    timed("merge + save", t0, scalable="query")
    log(f"[{c}] {stats['pairs']:,} pairs ({stats['pairs_per_query']:.0f}/query)"
        + (f"   recall old+B {stats['recall_old_b']*100:.2f}% -> all {stats['recall_all']*100:.2f}%"
           if MODE == "train" else ""))
    del out, parts, q8, p8, s8, n8, a8, P_rec, Q_rec
    gc.collect()
    mem(f"after {c}")

  [  1.0 min] [India] 265,098 queries vs 4,133,346 pool records


L8 pool India:   0%|          | 0/42 [00:00<?, ?chunk/s]

L8 queries India:   0%|          | 0/3 [00:00<?, ?chunk/s]

    [MEM] India L8 index               RAM=  8.4 GB  free= 23.9 GB


L8 search:   0%|          | 0/133 [00:00<?, ?chunk/s]

...

L8 pool US:   0%|          | 0/62 [00:00<?, ?chunk/s]

L8 queries US:   0%|          | 0/4 [00:00<?, ?chunk/s]

    [MEM] US L8 index                  RAM= 10.4 GB  free= 22.0 GB


L8 search:   0%|          | 0/199 [00:00<?, ?chunk/s]

  [ 27.0 min] [US] 39,714,913 pairs (100/query)   recall old+B 98.07% -> all 99.18%
    [MEM] after US                     RAM=  9.9 GB  free= 22.9 GB


In [9]:
# %% [CELL 8] Summary
print("\n" + "=" * 78)
print(f"PHASE C {MODE.upper()} {'SMOKE' if SMOKE else 'FULL'} SUMMARY")
print("=" * 78)
for c in COUNTRIES:
    s = json.load(open(OUT / c / "stats_c.json"))
    print(f"  {c:<8} queries {s['queries']:>9,}   pairs {s['pairs']:>11,} ({s['pairs_per_query']:.0f}/query)   "
          f"L8 {s['l8_pairs']:,}   L2t {s['l2t_pairs']:,}")
    if MODE == "train":
        print(f"           recall: old {s['recall_old']*100:6.2f}%   old+B {s['recall_old_b']*100:6.2f}%   "
              f"old+B+C {s['recall_all']*100:6.2f}%    held-out: {s['recall_old_b_heldout']*100:.2f}% -> "
              f"{s['recall_all_heldout']*100:.2f}%")
        print(f"           alone: L8 {s['recall_l8_alone']*100:.1f}%   L2t {s['recall_l2t_alone']*100:.1f}%   "
              f"new pairs not in old+B: {s['new_pairs_per_query']:.0f}/query")
if SMOKE:
    fq, fp = full_q / max(used_q, 1), full_p / max(used_p, 1)
    factor = {"pool": fp, "query": fq, "both": fq * fp}
    print(f"\n  ROUGH PROJECTION (queries x{fq:.0f}, pool x{fp:.0f}; fixed costs make it an overestimate)")
    tot = 0.0
    for stage, (sec, kind) in TIMES.items():
        full = sec * factor.get(kind, 1.0); tot += full
        print(f"    {stage:<22}{sec:>8.1f}s -> {full/60:7.1f} min")
    print(f"    {'TOTAL':<22}{'':>9}    {tot/60:7.1f} min")
    print("  Smoke uses a pool slice: recall above is NOT meaningful.")
    print("\n  CODE CHECK PASSED. Set SMOKE = False, then Save Version -> Save & Run All.")
log("PHASE C COMPLETE")


PHASE C TRAIN FULL SUMMARY
  India    queries   265,098   pairs  34,084,960 (129/query)   L8 26,508,594   L2t 7,952,940
           recall: old  90.01%   old+B  92.66%   old+B+C  98.23%    held-out: 92.65% -> 98.18%
           alone: L8 94.7%   L2t 16.7%   new pairs not in old+B: 98/query
  US       queries   397,165   pairs  39,714,913 (100/query)   L8 39,714,913   L2t 0
           recall: old  97.35%   old+B  98.07%   old+B+C  99.18%    held-out: 98.09% -> 99.18%
           alone: L8 96.7%   L2t 0.0%   new pairs not in old+B: 73/query
  [ 27.0 min] PHASE C COMPLETE


In [10]:
# %% [CELL 9] Clean-up: keep only ckpt_c_<mode>/ in the saved output
if not SMOKE:
    shutil.rmtree(WORK / "_unzipped", ignore_errors=True)
print("contents of /kaggle/working (what a committed version saves):")
for p in sorted(WORK.iterdir()):
    size = sum(f.stat().st_size for f in p.rglob("*") if f.is_file()) if p.is_dir() else p.stat().st_size
    print(f"  {p.name:<28} {size/1e9:6.2f} GB")

contents of /kaggle/working (what a committed version saves):
  .virtual_documents             0.00 GB
  ckpt_c_train                   1.25 GB
  ckpt_c_train_smoke             0.01 GB


In [1]:
print("hi")

hi


In [2]:
import shutil

shutil.make_archive(
    "/kaggle/working/ckpt_c_train",
    "zip",
    "/kaggle/working/ckpt_c_train"
)

print("Created: /kaggle/working/ckpt_c_train.zip")

Created: /kaggle/working/ckpt_c_train.zip


In [3]:
from IPython.display import FileLink

FileLink("/kaggle/working/ckpt_c_train.zip")

/kaggle/working/ckpt_c_train.zip

In [4]:
import os

size_gb = os.path.getsize("/kaggle/working/ckpt_c_train.zip") / (1024**3)
print(f"ZIP size: {size_gb:.2f} GB")

ZIP size: 0.44 GB


In [5]:
# Zip ckpt_c_<mode> (Phase C output) for download: one zip per country.
MODE = "train"          # "train" for ckpt_c_train, "test" for ckpt_c_test

import zipfile, time
from pathlib import Path

W = Path("/kaggle/working")
NAME = f"ckpt_c_{MODE}"
# 1) current session's /kaggle/working, else 2) an attached input (e.g. a committed notebook's output)
if (W / NAME).exists():
    SRC = W / NAME
else:
    hits = [p.parent.parent for p in Path("/kaggle/input").rglob("pairs_c.npz") if p.parent.parent.name == NAME]
    assert hits, f"{NAME} not found in /kaggle/working or /kaggle/input (attach the Phase C notebook output)"
    SRC = hits[0]
print("source:", SRC)

OUT = W / "_download"
OUT.mkdir(exist_ok=True)
for cdir in sorted(d for d in SRC.iterdir() if d.is_dir()):          # France / India / US
    files = sorted(f for f in cdir.iterdir() if f.is_file())
    dst = OUT / f"{NAME}_{cdir.name}.zip"
    t = time.time()
    with zipfile.ZipFile(dst, "w", zipfile.ZIP_DEFLATED, compresslevel=6) as z:
        for f in files:
            z.write(f, arcname=f"{NAME}/{cdir.name}/{f.name}")        # layout Stage 2 v3 expects
    with zipfile.ZipFile(dst) as z:
        ok = z.testzip() is None
    raw = sum(f.stat().st_size for f in files)
    print(f"{dst.name:<28} {raw/1e9:5.2f} GB -> {dst.stat().st_size/1e9:5.2f} GB   "
          f"{'OK' if ok else 'CORRUPT'}   ({time.time()-t:.0f}s)")

source: /kaggle/working/ckpt_c_train
ckpt_c_train_India.zip        0.58 GB ->  0.21 GB   OK   (28s)
ckpt_c_train_US.zip           0.68 GB ->  0.26 GB   OK   (29s)


In [6]:
import os
from IPython.display import FileLink, display

os.chdir("/kaggle/working")
for f in sorted(os.listdir("_download")):
    size = os.path.getsize(f"_download/{f}") / 1e9
    print(f"{f}  ({size:.2f} GB)")
    display(FileLink(f"_download/{f}"))

ckpt_c_train_India.zip  (0.21 GB)


/kaggle/working/_download/ckpt_c_train_India.zip

ckpt_c_train_US.zip  (0.26 GB)


/kaggle/working/_download/ckpt_c_train_US.zip